## 0.1. Auto-regressive model training loss

* 自回归模型，是词表粒度的多分类问题，用多分类问题的交叉熵定义其loss
    * 其形式为 (nll, negative log likelihood)：
      $$ L = -\frac{1}{N} \sum_{i=1}^{N} \log P(y_i) $$
    * LM head: one hot 分布（ground truth 分布）与预测概率分布的交叉熵；
        * 词表粒度的分类问题
    * 完全随机的情况下，对于 $|V| = 10000$ 时，其 $\log \frac{1}{10000} = 9.21$
* 二分类、多分类交叉熵

概念 1：自回归模型 (Auto-regressive model)什么是自回归？简单来说，就是“根据前文预测下一个字”。比如我说“白日依山...”，模型就要预测下一个字是“尽”。它是一步一步、一个词一个词地往外蹦的，这就叫自回归。为什么是“词表粒度的多分类问题”？假设模型的词典里一共有 10,000 个词（也就是词表大小 $\vert{}V\vert{} = 10000$）。当模型要预测下一个词时，它其实是在做一道有一万个选项的单项选择题。从多个选项中选一个，在机器学习中就叫做“多分类问题”。


损失函数 Loss 和 NLL (Negative Log Likelihood)
学生做错了题要扣分，模型预测错了也要有惩罚，这个惩罚机制就是 Loss（损失）。
图片中给出了公式：

$$L = -\frac{1}{N} \sum_{i=1}^{N} \log P(y_i)$$通俗拆解这个公式：$P(y_i)$：表示模型认为“正确答案”的概率是多少。比如正确答案是“尽”，模型如果预测“尽”的概率是 $0.9$（90%），说明它学得很好；如果预测概率只有 $0.01$（1%），说明它学得很烂。$\log$：对数操作。因为概率是小于1的数，连乘起来会非常非常小（容易导致计算机下溢出），取对数可以把乘法变成加法。负号 ($-$)：因为概率 $P$ 介于 0 和 1 之间，它的 $\log$ 值是个负数。为了让“惩罚（Loss）”变成一个正数（Loss越大说明越糟糕），所以前面要加个负号。这就叫 NLL（负对数似然）。$\frac{1}{N} \sum$：就是把所有题目的惩罚值加起来，求个平均分。

概念 3：LM head 与 交叉熵 (Cross Entropy)
LM head (Language Model Head)：这是大模型神经网络的最后一层。你可以把它理解为模型的“发声器官”或者“最终答题卡”，它负责输出那一万个词的概率分布。

Ground truth (真实分布) vs 预测分布：

Ground truth (one hot分布)：标准答案。比如正确答案是第3个词，那标准答案的概率分布就是 [0, 0, 1, 0, 0, ...]（只有正确词是1，其他全是0，这叫 one-hot）。

预测分布：模型自己瞎猜的。比如 [0.1, 0.05, 0.7, 0.15, 0, ...]。

交叉熵：就是用来对比这两种分布有多大差异的尺子。模型的预测分布越接近标准答案的 one-hot 分布，交叉熵（也就是 Loss）就越小，模型就越聪明。

概念 4：一个极其重要的常识——“完全随机情况下的 Loss 是 9.21”幻灯片里专门提到了：对于 |V| = 10000 时，其 log(1/10000) = 9.21这是什么意思？这是深度学习中非常有名的“初始损失值检查（Sanity Check）”。想象一个模型刚出生，什么都没学过。当它做这道一万个选项的选择题时，它只能纯蒙（完全随机）。蒙对正确答案的概率是多少？是 $\frac{1}{10000}$。根据上面的公式，此时的 Loss = $-\ln(\frac{1}{10000}) \approx 9.21$。（注：这里的 log 默认是自然对数 ln）。作用：程序员在开始训练模型时，如果看到屏幕上打印的第一步的 Loss 大约是 9.2左右，他就会松一口气，因为这说明代码写对了，模型确实是从“完全不懂”的白纸状态开始学习的。如果初始 Loss 是 100 或者 2，说明代码里肯定有 Bug。

* 二分类

  $$ L = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log P(\hat{y}_i) + (1 - y_i) \log(1 - P(\hat{y}_i)) \right] $$
 公式通俗拆解：$$y_i \log P(\hat{y}_i) + (1 - y_i) \log(1 - P(\hat{y}_i))$$这个公式设计得非常巧妙，你可以把它看作是一个“智能双联开关”：$y_i$：是标准答案（Ground Truth）。只能是 1 (是) 或者 0 (否)。$P(\hat{y}_i)$：是模型预测“是”的概率。比如模型觉得有 80% 的概率是猫，那这就是 0.8。这个开关是怎么工作的？当标准答案是“是” ($y_i = 1$) 时：公式的后半部分 (1 - 1) 变成了 0，整个后半部分就被消灭了。只剩下前半部分：$\log P(\hat{y}_i)$。模型预测的概率越接近 1，惩罚（Loss）就越小。当标准答案是“否” ($y_i = 0$) 时：公式的前半部分 0 * ... 变成了 0，前半部分被消灭了。只剩下后半部分：$\log(1 - P(\hat{y}_i))$。模型预测“是”的概率越低，说明它预测“否”的概率越高，惩罚（Loss）就越小。总结： 这个公式利用乘以 1 和 0 的数学技巧，把两种情况完美地揉进了同一个等式里。

* 多分类

  $$ L = -\frac{1}{N} \sum_{i=1}^{N} y_{i,c} \log P(\hat{y}_{i,c}) $$

  公式通俗拆解：$$y_{i,c} \log P(\hat{y}_{i,c})$$$y_{i,c}$：代表在第 $i$ 个样本中，第 $c$ 个类别的标准答案。因为是单选题，所以只有正确选项的值是 1，其他成千上万个错误选项的值全都是 0（这叫 One-hot 编码）。$P(\hat{y}_{i,c})$：模型预测这是第 $c$ 个类别的概率。为什么这个公式看起来比二分类短？因为既然只有正确答案对应的 $y_{i,c}$ 是 1，其他全为 0。那么在计算惩罚时，所有错误选项算出来的结果都是 0（直接被消灭了）。所以，系统根本不关心模型给错误选项打了多少分，它只盯着“模型给正确答案分配了多大的概率”。这和我们上一张图看到的那个公式本质上是一模一样的：$$L = -\frac{1}{N} \sum_{i=1}^{N} \log P(y_i)$$（因为 $y_{i,c} = 1$，所以省略不写了，只保留了正确选项的概率 $P$ 取对数）。

## 0.2. PPL

PPL: perplexity

* language model 好坏的评估指标
    * 较低的困惑度指模型的预测更加准确。
    
  $$ PPL = \exp \left( -\frac{1}{N} \sum_{i=1}^{N} \log P(y_i) \right) $$

* loss of ar model
  
  $$ L = \log PPL $$
    
    * minimize L == minimize PPL

概念 1：什么是 PPL (困惑度)?PPL 全称是 Perplexity。大白话讲，它衡量的是模型在预测下一个词时，到底有多“懵逼”或“困惑”。困惑度越低：说明模型对接下来的词心里越有底，预测越准确（不困惑）。困惑度越高：说明模型觉得接下来的词有无数种可能，完全在瞎猜（很困惑）。直观理解：如果一个模型的 PPL 是 10，大致可以理解为，模型在做选择题时，觉得有 10 个选项看起来都像是正确答案，它在 10 个词里犹豫不决。显然，在 10 个词里犹豫，比在 10000 个词里犹豫（第一张图提到的完全随机状态）要聪明得多。概念 2：PPL 怎么算？它和 Loss 有什么关系？你看 PPT 里的第一个大公式：$$ PPL = \exp \left( -\frac{1}{N} \sum_{i=1}^{N} \log P(y_i) \right) $$仔细观察括号里面的内容：-(1/N) * sum(log P(yi))。这不就是我们第一张图里讲过的交叉熵 Loss（损失）公式吗！所以，这个看似复杂的公式可以简化成极其简单的一句话：$$ PPL = \exp(Loss) $$ （即 PPL 等于自然常数 $e$ 的 Loss 次方）。概念 3：Loss 和 PPL 为什么可以互相推导？PPT 的下半部分给出了反向推导公式：既然 $PPL = e^{Loss}$，那么对等式两边同时取自然对数（$\log$），就变成了：$$ L = \log(PPL) $$那既然两者可以互相推导，为什么还要搞两个名字？Loss (交叉熵) 是给计算机看的。计算机在做微积分、通过梯度下降来学习时，相加的算式（对数把连乘变成了连加）更好算，而且不会因为概率相乘数值太小而导致计算机奔溃（下溢出）。PPL (困惑度) 是给人看的。给老板汇报时，你说“我们的 Loss 降到了 2.3”，别人很难有直观概念；但你说“模型的困惑度降到了 10（模型基本只在 10 个词里纠结）”，人类就非常容易理解这个模型的水平有多高。结论：minimize L == minimize PPL因为指数函数（$e^x$）和对数函数（$\log(x)$）在数学上都是“单调递增”的。水涨船高，水落船低。所以，让 Loss 变小（minimize L）和让困惑度变小（minimize PPL）在数学目标上是完全等价的。程序员在写代码让模型不断减小 Loss 时，实际上就是在让模型变得越来越不困惑。